# ChatGPMe Colab Upload Runner

This notebook uses the minimal `colab_upload.zip` package for paper-aligned style training.

Expected files after unzip:
- `style_train_2k.jsonl`
- `train_lora.py`
- `generate_with_adapter.py`
- `remote_inference_server.py`
- `requirements.txt`


## 1. Enable GPU

In Colab, set `Runtime > Change runtime type > GPU` before training.

In [1]:
!nvidia-smi || true

Wed Mar 25 14:48:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Upload and unzip `colab_upload.zip`

If you already uploaded and unzipped it manually, skip the upload cell.

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
!rm -rf colab_upload tinyllama-style-lora-colab tinyllama-style-lora-colab.zip
!unzip -o colab_upload.zip -d colab_upload
%cd colab_upload
!ls -la

## 3. Install dependencies

In [ ]:
!python -m pip install -U pip
!python -m pip install torch transformers datasets peft accelerate sentencepiece

## 4. Confirm the package and training recipe

In [ ]:
!ls -lh
!wc -l style_train_2k.jsonl
!grep -n "learning-rate\|max-length\|build_prompt_prefix\|instruction" train_lora.py || true

## 5. Train the style adapter

This follows the paper more closely: raw text, next-token objective, shorter sequences, lower learning rate.

In [ ]:
!python train_lora.py \
  --model-name TinyLlama/TinyLlama-1.1B-Chat-v1.0 \
  --dataset-path style_train_2k.jsonl \
  --output-dir tinyllama-style-lora-colab \
  --epochs 3 \
  --batch-size 2 \
  --grad-accum 2 \
  --max-length 256 \
  --learning-rate 5e-5 \
  --save-steps 50

## 6. Test raw continuation generation

The style adapter is trained for continuation, not chat instructions, so prompts here should be opening text rather than requests.

In [ ]:
!python generate_with_adapter.py \
  --model-name TinyLlama/TinyLlama-1.1B-Chat-v1.0 \
  --adapter-path tinyllama-style-lora-colab \
  --prompt "I have been thinking lately about how strange it is to build something important with two other people," \
  --max-new-tokens 120

In [ ]:
!python generate_with_adapter.py \
  --model-name TinyLlama/TinyLlama-1.1B-Chat-v1.0 \
  --adapter-path tinyllama-style-lora-colab \
  --prompt "When I sat down to write this application, I realized that the hardest part was not describing what I had done," \
  --max-new-tokens 120

## 7. Inspect and download artifacts

In [ ]:
!find tinyllama-style-lora-colab -maxdepth 2 -type f | sort

## 8. Optional: serve the model for the local UI

This starts a small HTTP server in Colab and exposes it through ngrok so your local app can call the GPU-backed model.

In [ ]:
!python -m pip install pyngrok

In [ ]:
!python remote_inference_server.py \
  --model-name TinyLlama/TinyLlama-1.1B-Chat-v1.0 \
  --adapter-path tinyllama-style-lora-colab \
  --port 8001 &

In [ ]:
from pyngrok import ngrok
public_url = ngrok.connect(8001)
print(public_url)

Run the local UI with:

```bash
CHATGPME_REMOTE_API=<public url> .venv/bin/python scripts/run_app.py
```

In [ ]:
!zip -r tinyllama-style-lora-colab.zip tinyllama-style-lora-colab

In [ ]:
from google.colab import files
files.download('tinyllama-style-lora-colab.zip')